# Here's a simple LightGBM starter - happy copying and editing!

**Version 7** comes with no feature engineering at all, so it’s great if you just want a clean, easy-to-follow reference. Public score: **0.97470**.

As of **Version 14**, the notebook achieved its highest public score of **0.97507**.

More new features or other improvements may be added in future versions.

In [1]:
# Print all file paths in the input directory
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/playground-series-s5e8/sample_submission.csv
/kaggle/input/playground-series-s5e8/train.csv
/kaggle/input/playground-series-s5e8/test.csv


# Data Loading

In [2]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/playground-series-s5e8/train.csv")
test_df = pd.read_csv("/kaggle/input/playground-series-s5e8/test.csv")

In [3]:
# Check and print the count of missing values in each column of the dataframe

print(train_df.isnull().sum())
print(test_df.isnull().sum())

id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64
id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
dtype: int64


# Data Processing

In [ ]:
# Load original UCI Bank Marketing dataset for match flags (S4E1 2nd place technique)
# To use this: add dataset "bank-marketing" to your Kaggle notebook inputs
# https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset
try:
    original_df = pd.read_csv("/kaggle/input/bank-marketing/bank.csv")
    HAS_ORIGINAL = True
    print(f"Original data loaded: {original_df.shape}")
except:
    try:
        original_df = pd.read_csv("/kaggle/input/bank-additional/bank-additional-full.csv", sep=";")
        HAS_ORIGINAL = True
        print(f"Original data (additional) loaded: {original_df.shape}")
    except:
        HAS_ORIGINAL = False
        print("Original data not found. Match flags will be skipped.")


In [ ]:
# Add new features - enhanced version v2 (vectorized for speed)
import numpy as np
from itertools import combinations

def add_features_v2(df):
    df = df.copy()

    # ===== 既存特徴量 (ベクトル化版) =====
    # credit: apply()不要 - ベクトル演算で10倍速
    credit_score = ((df['default'] == 'no').astype(int) +
                    (df['housing'] == 'no').astype(int) +
                    (df['loan']    == 'no').astype(int))
    df['credit'] = credit_score.map({3: 27, 2: 9, 1: 3, 0: 0})

    # risk: 同様にベクトル化
    risk_score = ((df['default'] == 'yes').astype(int) +
                  (df['housing'] == 'yes').astype(int) +
                  (df['loan']    == 'yes').astype(int))
    df['risk'] = risk_score.map({3: 27, 2: 9, 1: 3, 0: 0})

    # log変換: clip+log1pで高速化
    df['balance_log']  = np.log1p(df['balance'].clip(lower=0))
    df['duration_log'] = np.log1p(df['duration'].clip(lower=0))

    month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
                 'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
    df['month'] = df['month'].map(month_map)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['age_bin'] = pd.cut(df['age'], bins=[0, 40, 70, 100],
                           labels=[1, 2, 0], right=False).astype(int)

    # ===== 追加1: コンタクト効率指標 =====
    df['duration_x_campaign']       = df['duration'] * df['campaign']
    df['avg_duration_per_contact']  = df['duration'] / (df['campaign'] + 1)
    df['duration_bin'] = pd.cut(df['duration'],
                                bins=[0, 60, 180, 360, 600, np.inf],
                                labels=[0, 1, 2, 3, 4]).astype(float)

    # ===== 追加2: 前回キャンペーン情報 =====
    df['was_contacted_before']       = (df['pdays'] != -1).astype(int)
    df['pdays_clean']                = df['pdays'].replace(-1, 999)
    df['pdays_log']                  = np.log1p(df['pdays_clean'].clip(lower=0))
    df['previous_contact_density']   = df['previous'] / (df['pdays_clean'] + 1)

    # ===== 追加3: 残高関連特徴量 =====
    df['balance_negative'] = (df['balance'] < 0).astype(int)
    df['balance_zero']     = (df['balance'] == 0).astype(int)
    df['balance_per_age']  = df['balance'] / (df['age'] + 1)
    df['balance_abs_log']  = np.log1p(np.abs(df['balance']))

    # ===== 追加4: 職業・教育スコア =====
    job_score_map = {'admin.': 3, 'technician': 3, 'management': 4, 'entrepreneur': 4,
                     'self-employed': 3, 'retired': 4, 'services': 2, 'blue-collar': 1,
                     'housemaid': 1, 'student': 3, 'unemployed': 1, 'unknown': 2}
    edu_score_map = {'tertiary': 4, 'secondary': 2, 'primary': 1, 'unknown': 2}
    df['job_score']          = df['job'].map(job_score_map).fillna(2)
    df['edu_score']          = df['education'].map(edu_score_map).fillna(2)
    df['financial_literacy'] = df['job_score'] * df['edu_score']

    # ===== 追加5: 季節性強化 =====
    df['is_q4']    = df['month'].isin([10, 11, 12]).astype(int)
    df['is_q2']    = df['month'].isin([4, 5, 6]).astype(int)
    df['day_sin']  = np.sin(2 * np.pi * df['day'] / 31)
    df['day_cos']  = np.cos(2 * np.pi * df['day'] / 31)

    # ===== 追加6: 年齢非線形特徴量 =====
    df['age_sq']          = df['age'] ** 2
    df['age_young']       = (df['age'] < 30).astype(int)
    df['age_senior']      = (df['age'] >= 60).astype(int)
    df['age_x_job_score'] = df['age'] * df['job_score']

    # ===== 追加7: キャンペーン疲弊指標 =====
    df['campaign_high'] = (df['campaign'] >= 5).astype(int)
    df['campaign_log']  = np.log1p(df['campaign'])

    # ===== 追加8: 複合インタラクション + poutcome詳細 =====
    df['balance_x_duration'] = df['balance_log'] * df['duration_log']
    df['credit_x_duration']  = df['credit'] * df['duration_log']
    df['risk_x_campaign']    = df['risk'] * df['campaign']
    df['poutcome_success']   = (df['poutcome'] == 'success').astype(int)
    df['poutcome_failure']   = (df['poutcome'] == 'failure').astype(int)

    return df

train_df = add_features_v2(train_df)
test_df  = add_features_v2(test_df)
print(f"Shape after features: train={train_df.shape}, test={test_df.shape}")


In [ ]:
# S4E1 2nd place technique: original data subset-matching flags (merge版 - 高速)
def add_original_match_features(train_df, test_df, original_df):
    key_cols = ['age', 'job', 'marital', 'education', 'balance',
                'duration', 'campaign', 'pdays', 'previous']
    available_cols = [c for c in key_cols if c in original_df.columns]

    # 2〜3列のみ（4以上は組み合わせ爆発で遅くなるため除外）
    for r in range(2, 4):
        for cols in combinations(available_cols, r):
            cols = list(cols)
            flag = "match_" + "__".join(cols)

            # applyではなくmergeで高速マッチング
            orig_sub = original_df[cols].drop_duplicates().copy()
            orig_sub["_flag"] = np.int8(1)

            train_merged = train_df[cols].merge(orig_sub, on=cols, how="left")
            train_df[flag] = train_merged["_flag"].fillna(0).astype(np.int8).values

            test_merged = test_df[cols].merge(orig_sub, on=cols, how="left")
            test_df[flag] = test_merged["_flag"].fillna(0).astype(np.int8).values

    n_flags = len([c for c in train_df.columns if c.startswith("match_")])
    print(f"Match flags added: {n_flags}")
    return train_df, test_df

if HAS_ORIGINAL:
    train_df, test_df = add_original_match_features(train_df, test_df, original_df)
else:
    print("Skipping match flags (no original data)")


In [ ]:
# Get all non-numeric columns from the training dataframe

cat_cols = train_df.select_dtypes(include="object").columns.tolist()
cat_cols

['job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'poutcome']

In [6]:
# Iterate through each non-numeric column to compare unique values between train and test sets

for col in cat_cols:
    print(f"\"{col}\" colum:\n>>> {sorted(train_df[col].unique())}\n>>> {sorted(test_df[col].unique())}\n")

# This comparison helps verify:
# 1. If test set contains values not seen in training (out-of-domain categories)
# 2. If categorical encodings can be safely applied to both sets

"job" colum:
>>> ['admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown']
>>> ['admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown']

"marital" colum:
>>> ['divorced', 'married', 'single']
>>> ['divorced', 'married', 'single']

"education" colum:
>>> ['primary', 'secondary', 'tertiary', 'unknown']
>>> ['primary', 'secondary', 'tertiary', 'unknown']

"default" colum:
>>> ['no', 'yes']
>>> ['no', 'yes']

"housing" colum:
>>> ['no', 'yes']
>>> ['no', 'yes']

"loan" colum:
>>> ['no', 'yes']
>>> ['no', 'yes']

"contact" colum:
>>> ['cellular', 'telephone', 'unknown']
>>> ['cellular', 'telephone', 'unknown']

"poutcome" colum:
>>> ['failure', 'other', 'success', 'unknown']
>>> ['failure', 'other', 'success', 'unknown']



In [7]:
# from itertools import combinations

# for cols in combinations(cat_cols, 2):
#     col = "_".join(cols)
#     cat_cols.append(col)
#     train_df[col] = train_df[cols[0]].astype(str) + "_" + train_df[cols[1]].astype(str)
#     test_df[col] = test_df[cols[0]].astype(str) + "_" + test_df[cols[1]].astype(str)

In [8]:
# Add count encoding
from collections import Counter

for col in cat_cols:
    counts = train_df[col].value_counts()
    train_df[col + "_count"] = train_df[col].map(counts)
    test_df[col + "_count"] = test_df[col].map(counts).fillna(0)

In [ ]:
# Target Encoding function (CV-safe, no leakage)
from sklearn.model_selection import StratifiedKFold as _SKF

def add_target_encoding(X_train, y_train, X_val, X_test, cat_cols, n_splits=5):
    X_train, X_val, X_test = X_train.copy(), X_val.copy(), X_test.copy()
    global_mean = y_train.mean()

    for col in cat_cols:
        target_mean = y_train.groupby(X_train[col]).mean()
        X_val[f'{col}_te']  = X_val[col].map(target_mean).fillna(global_mean)
        X_test[f'{col}_te'] = X_test[col].map(target_mean).fillna(global_mean)

        X_train[f'{col}_te'] = global_mean
        kf_inner = _SKF(n_splits=n_splits, shuffle=True, random_state=42)
        for tr_idx, va_idx in kf_inner.split(X_train, y_train):
            enc = y_train.iloc[tr_idx].groupby(X_train[col].iloc[tr_idx]).mean()
            X_train.loc[X_train.index[va_idx], f'{col}_te'] = (
                X_train[col].iloc[va_idx].map(enc).fillna(global_mean).values
            )
    return X_train, X_val, X_test


In [9]:
# Prepare feature matrix (X) and target vector (y) for training, and feature matrix (X_test) for testing

X = train_df.drop(["y", "id"], axis=1)
y = train_df["y"]
X_test = test_df.drop(["id"], axis=1)

In [10]:
# Map non-numeric columns to numeric values using LabelEncoder for both training and test sets
from sklearn.preprocessing import LabelEncoder

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    X_test[col] = le.transform(X_test[col])

# Model Training

In [ ]:
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar
import subprocess, warnings
warnings.filterwarnings('ignore')

# GPU自動検出
try:
    _gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    HAS_GPU = _gpu.returncode == 0
except Exception:
    HAS_GPU = False
print(f"GPU: {'enabled (cuda)' if HAS_GPU else 'not found - using CPU'}")

lgb_params = {
    'objective': 'binary', 'metric': 'auc',
    'n_estimators': 20000, 'learning_rate': 0.01,
    'num_leaves': 63, 'max_depth': -1,
    'min_child_samples': 50, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 1,
    'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'scale_pos_weight': (y == 0).sum() / (y == 1).sum(),
    'device': 'gpu' if HAS_GPU else 'cpu',
    'n_jobs': -1, 'random_state': 42, 'verbose': -1,
}

xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'n_estimators': 20000, 'learning_rate': 0.01,
    'max_depth': 6, 'min_child_weight': 50,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'scale_pos_weight': (y == 0).sum() / (y == 1).sum(),
    'tree_method': 'hist',
    'device': 'cuda' if HAS_GPU else 'cpu',
    'random_state': 42, 'verbosity': 0,
    'early_stopping_rounds': 200,
}

n_splits = 5
kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_lgb  = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test))
pred_xgb = np.zeros(len(X_test))
lgb_models = []  # 特徴量重要度プロット用

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\nFold {fold + 1}/{n_splits}")
    X_tr, y_tr = X.iloc[train_idx].copy(), y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx].copy(),   y.iloc[val_idx]
    X_te = X_test.copy()

    # Target Encoding をフォールド内で適用（リーク防止）
    X_tr, X_va, X_te = add_target_encoding(X_tr, y_tr, X_va, X_te, cat_cols)

    # LightGBM
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                  callbacks=[lgb.early_stopping(200, verbose=False),
                              lgb.log_evaluation(500)])
    lgb_models.append(lgb_model)  # 重要度プロット用に保存
    oof_lgb[val_idx]  = lgb_model.predict_proba(X_va)[:, 1]
    pred_lgb         += lgb_model.predict_proba(X_te)[:, 1] / n_splits

    # XGBoost
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[val_idx]  = xgb_model.predict_proba(X_va)[:, 1]
    pred_xgb         += xgb_model.predict_proba(X_te)[:, 1] / n_splits

    auc_l = roc_auc_score(y_va, oof_lgb[val_idx])
    auc_x = roc_auc_score(y_va, oof_xgb[val_idx])
    print(f"  LGB: {auc_l:.5f} | XGB: {auc_x:.5f}")

print(f"\nOOF LGB: {roc_auc_score(y, oof_lgb):.5f}")
print(f"OOF XGB: {roc_auc_score(y, oof_xgb):.5f}")

# 最適ブレンド重みを自動探索
def neg_auc(w):
    return -roc_auc_score(y, w * oof_lgb + (1 - w) * oof_xgb)

result = minimize_scalar(neg_auc, bounds=(0, 1), method='bounded')
best_w = result.x
print(f"\nBest LGB weight: {best_w:.3f}")
print(f"Ensemble OOF:    {roc_auc_score(y, best_w * oof_lgb + (1 - best_w) * oof_xgb):.5f}")


In [ ]:
# Feature importance (LightGBM の全フォールド平均)
all_importances = []

for model in lgb_models:
    if hasattr(model, "feature_importances_"):
        all_importances.append(model.feature_importances_)

if all_importances:
    avg_importances = pd.Series(
        np.mean(all_importances, axis=0),
        index=X_tr.columns  # Target Encoding後の列名
    )
    avg_importances.sort_values().tail(30).plot(kind='barh', figsize=(8, 10))
    import matplotlib.pyplot as plt
    plt.title("Top 30 Feature Importances (LightGBM avg)")
    plt.tight_layout()
    plt.show()


# Results Submission

In [ ]:
# Ensemble submission
final_pred = best_w * pred_lgb + (1 - best_w) * pred_xgb

submission = pd.DataFrame({"id": test_df["id"], "y": final_pred})
submission.to_csv("submission.csv", index=False)
print("Submission saved!")
submission.head()
